# 🌊 AquaYOLO: Underwater Object Detection & Benchmark Training

This notebook runs training, evaluation, and complete metric visualization for the **AquaYOLO** marine object detection project in Google Colab with GPU acceleration.

### 📌 Features in this Notebook:
- **Multi-Zip Dataset Support**: Automatically extracts and merges your dataset from **three (or more) zip files** stored in Google Drive.
- **High-Accuracy YOLO Training**: Transfer learning with marine and COCO pretrained backbones, mosaic augmentation, and cosine learning rate schedule.
- **Metric Visualizations**: Displays all YOLO-generated metric graphs directly in notebook outputs:
  - `results.png` (Train/Val loss, Precision, Recall, mAP@0.50, mAP@0.50:0.95)
  - `confusion_matrix.png` and `confusion_matrix_normalized.png`
  - `PR_curve.png` (Precision-Recall curve)
  - `F1_curve.png` (F1 Score vs Confidence)
  - `P_curve.png` and `R_curve.png`
  - `val_batch0_pred.jpg` (Visual prediction comparisons)
- **AquaYOLO Custom Attention**: Run and evaluate the custom AquaYOLO model featuring SAC (Spectral Attenuation Compensator) and TCG (Turbidity Contrast Gate).

## 🚀 Step 1: Check GPU Runtime
Make sure you have selected **GPU** as your runtime type (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi

## 📂 Step 2: Mount Google Drive
Mount your Google Drive to load your dataset zip files.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 📦 Step 3: Clone AquaYOLO Repository & Install Dependencies

In [ ]:
# Clone the repository from GitHub
!git clone https://github.com/Aravind8833/AquaYOLO.git
%cd /content/AquaYOLO

# Install dependencies
!pip install -q ultralytics opencv-python matplotlib seaborn pyyaml pandas pillow tqdm
print('\n Environment ready!')

## 📁 Step 4: Extract Dataset from Multiple (3) Zip Files in Google Drive

If your dataset is split into **three zip files** (e.g. `part1.zip`, `part2.zip`, `part3.zip` OR `train.zip`, `val.zip`, `test.zip`), you can either:
1. **Option A**: Provide the exact paths to your 3 zip files in `ZIP_FILE_PATHS` below, **OR**
2. **Option B**: Provide the Google Drive folder that contains them in `DRIVE_FOLDER_WITH_ZIPS` (it will auto-detect all `.zip` files).

The code extracts and organizes all files into `/content/dataset`, auto-creates/resolves `data.yaml`, and prints an audit of all images.

In [ ]:
import os
import shutil
import zipfile
from pathlib import Path
import yaml
from tqdm import tqdm

# =====================================================================
# ⚙️ CONFIGURE YOUR THREE ZIP FILES HERE (UPDATE THESE PATHS):
# =====================================================================
# OPTION A: List the 3 zip files explicitly:
ZIP_FILE_PATHS = [
    "/content/drive/MyDrive/dataset_part1.zip",
    "/content/drive/MyDrive/dataset_part2.zip",
    "/content/drive/MyDrive/dataset_part3.zip",
]

# OPTION B (Alternative): Or point to the Drive folder containing your zips:
DRIVE_FOLDER_WITH_ZIPS = "/content/drive/MyDrive/AquaYOLO_zips"
# =====================================================================

LOCAL_DATASET_DIR = Path("/content/dataset")
LOCAL_DATASET_DIR.mkdir(parents=True, exist_ok=True)

# 1. Collect zip files to extract
zips_to_extract = []
for p in ZIP_FILE_PATHS:
    pth = Path(p)
    if pth.is_file() and pth.suffix.lower() == ".zip":
        zips_to_extract.append(pth)

# If list paths were not found, check the folder:
if not zips_to_extract and Path(DRIVE_FOLDER_WITH_ZIPS).is_dir():
    zips_to_extract = sorted(Path(DRIVE_FOLDER_WITH_ZIPS).glob("*.zip"))

print(f"Found {len(zips_to_extract)} zip file(s) to extract:")
for z in zips_to_extract:
    size_mb = z.stat().st_size / (1024 * 1024)
    print(f"  📦 {z.name} ({size_mb:.1f} MB)")

if not zips_to_extract:
    raise FileNotFoundError(
        "⚠️ No zip files found! Please check ZIP_FILE_PATHS or DRIVE_FOLDER_WITH_ZIPS above."
    )

# 2. Extract each zip file
for zpath in zips_to_extract:
    print(f"\nExtracting {zpath.name} to {LOCAL_DATASET_DIR}...")
    with zipfile.ZipFile(zpath, 'r') as zf:
        members = zf.namelist()
        for m in tqdm(members, desc=f"  {zpath.stem}", unit="file"):
            zf.extract(m, LOCAL_DATASET_DIR)

print("\n All zip files extracted successfully!")

# 3. Standardize structure if nested (ensure images/ and labels/ are located)
def find_split_dir(base_dir, split_name, subfolder_name):
    # Look for paths like base/**/images/train or base/**/train/images
    candidates = list(base_dir.rglob(f"**/{subfolder_name}/{split_name}"))
    if candidates:
        return candidates[0]
    candidates2 = list(base_dir.rglob(f"**/{split_name}/{subfolder_name}"))
    if candidates2:
        return candidates2[0]
    return None

# Standard target folders
target_images = LOCAL_DATASET_DIR / "images"
target_labels = LOCAL_DATASET_DIR / "labels"

for split in ["train", "val", "test"]:
    t_img = target_images / split
    t_lbl = target_labels / split
    t_img.mkdir(parents=True, exist_ok=True)
    t_lbl.mkdir(parents=True, exist_ok=True)

    found_img = find_split_dir(LOCAL_DATASET_DIR, split, "images")
    if found_img and found_img.resolve() != t_img.resolve():
        for f in found_img.glob("*.*"):
            dest = t_img / f.name
            if not dest.exists():
                shutil.move(str(f), str(dest))

    found_lbl = find_split_dir(LOCAL_DATASET_DIR, split, "labels")
    if found_lbl and found_lbl.resolve() != t_lbl.resolve():
        for f in found_lbl.glob("*.txt"):
            dest = t_lbl / f.name
            if not dest.exists():
                shutil.move(str(f), str(dest))

# 4. Locate or generate data.yaml
DATA_YAML_PATH = LOCAL_DATASET_DIR / "data.yaml"
classes = [
    "jellyfish", "penguin", "puffin", "starfish", "stingray",
    "basking_shark", "blacktip_reef_shark", "great_white_shark",
    "hammerhead_shark", "nurse_shark", "oceanic_whitetip_shark",
    "thresher_shark", "whale_shark", "whitetip_reef_shark",
    "ballan_wrasse", "cuckoo_wrasse", "corkwing_wrasse",
    "goldsinny_wrasse", "rock_cook", "atlantic_cod",
    "pollack", "whiting", "spiny_dogfish", "brown_crab",
    "two_spot_goby", "human"
]
data_dict = {
    "path": str(LOCAL_DATASET_DIR.resolve()),
    "train": "images/train",
    "val": "images/val",
    "test": "images/test",
    "names": {i: name for i, name in enumerate(classes)}
}
with open(DATA_YAML_PATH, "w", encoding="utf-8") as f:
    yaml.safe_dump(data_dict, f, sort_keys=False)

# 5. Print dataset verification summary
print("=" * 60)
print("DATASET VERIFICATION AUDIT:")
print("=" * 60)
for s in ["train", "val", "test"]:
    n_img = len(list((target_images / s).glob("*.*")))
    n_lbl = len(list((target_labels / s).glob("*.txt")))
    print(f"  {s.upper():5s} split -> {n_img:5d} images | {n_lbl:5d} label files")
print("=" * 60)
print(f"Config saved: {DATA_YAML_PATH}")

## 🏋️ Step 5: High-Accuracy Training (YOLO Benchmark)
Trains YOLO with transfer learning, cosine learning rate decay, and mosaic augmentation to achieve high accuracy (>75% mAP50).

In [ ]:
# Run high-accuracy training
!python train_yolo.py \
    --data "{DATA_YAML_PATH}" \
    --model yolov8s.pt \
    --epochs 50 \
    --batch 16 \
    --imgsz 640 \
    --project /content/AquaYOLO/runs/yolo_train \
    --name colab_run

## 📊 Step 6: Display All Generated Metric Graphs
This displays every single metric graph generated by YOLO directly in the notebook output:
1. **Training & Validation Loss / Metrics Curve** (`results.png`)
2. **Confusion Matrix & Normalized Confusion Matrix** (`confusion_matrix.png`)
3. **Precision-Recall Curve** (`PR_curve.png`)
4. **F1-Confidence Curve** (`F1_curve.png`)
5. **Precision & Recall Curves** (`P_curve.png`, `R_curve.png`)
6. **Validation Ground Truth vs Model Predictions** (`val_batch0_labels.jpg`, `val_batch0_pred.jpg`)

In [ ]:
from IPython.display import Image, display
import matplotlib.pyplot as plt
from pathlib import Path
import pandas as pd

RUN_DIR = Path("/content/AquaYOLO/runs/yolo_train/colab_run")

def show_graph(image_filename, title):
    p = RUN_DIR / image_filename
    if p.exists():
        print("=" * 75)
        print(f"{title}: {p.name}")
        print("=" * 75)
        display(Image(filename=str(p), width=900))
    else:
        print(f"{image_filename} not generated yet.")

# 1. Training & Validation Results (Losses, Precision, Recall, mAP50, mAP50-95)
show_graph("results.png", "TRAINING & VALIDATION METRICS OVER EPOCHS")

# 2. Precision-Recall Curve
show_graph("PR_curve.png", "PRECISION-RECALL (PR) CURVE")

# 3. F1-Confidence Curve
show_graph("F1_curve.png", "F1-SCORE VS CONFIDENCE CURVE")

# 4. Precision & Recall vs Confidence Curves
show_graph("P_curve.png", "PRECISION VS CONFIDENCE CURVE")
show_graph("R_curve.png", "RECALL VS CONFIDENCE CURVE")

# 5. Confusion Matrices
show_graph("confusion_matrix.png", "CONFUSION MATRIX")
show_graph("confusion_matrix_normalized.png", "NORMALIZED CONFUSION MATRIX")

# 6. Visual Detections on Validation Set (Ground Truth vs Model Prediction)
show_graph("val_batch0_labels.jpg", "GROUND TRUTH LABELS")
show_graph("val_batch0_pred.jpg", "MODEL PREDICTIONS ON VALIDATION BATCH")

## 📋 Step 7: View Numerical Metrics Table (Epoch by Epoch)

In [ ]:
results_csv = RUN_DIR / "results.csv"
if results_csv.exists():
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    print("\n--- FINAL EPOCHS SUMMARY ---")
    display(df.tail(10))
else:
    print("results.csv not found.")

## 🧠 Step 8: (Optional) Train Custom AquaYOLO Architecture
AquaYOLO model with CSPDarknet backbone, Spectral Attenuation Compensator (SAC), and Turbidity Contrast Gate (TCG).

In [ ]:
# Train custom AquaYOLO model
!python src/aquayolo/train.py \
    --data "{DATA_YAML_PATH.parent}" \
    --epochs 30 \
    --batch 4 \
    --size 640 \
    --lr 0.0001 \
    --out ./runs/aquayolo_custom_colab

# Show AquaYOLO loss curves
aqua_loss_plot = Path("./runs/aquayolo_custom_colab/loss_curve.png")
if aqua_loss_plot.exists():
    display(Image(filename=str(aqua_loss_plot), width=800))
aqua_comp_plot = Path("./runs/aquayolo_custom_colab/loss_components.png")
if aqua_comp_plot.exists():
    display(Image(filename=str(aqua_comp_plot), width=900))